In [3]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

Starting virtual X frame buffer: Xvfb.


In [10]:
%pip install gymnasium[atari]

Note: you may need to restart the kernel to use updated packages.


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



%pip install "gymnasium[atari]"

In [16]:
import numpy as np
import gymnasium as gym
import ale_py
from atari_wrappers import nature_dqn_env
import psutil
gym.register_envs(ale_py)


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = psutil.cpu_count()  # change this if you have more than 8 CPU ;)
print("Number of CPU: ", nenvs)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


Number of CPU:  32


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [44]:
env.action_space.n

np.int64(6)

In [35]:
# import tensorflow as torch
# import torch as tf
import torch
import torch.nn.init

def init_orthogonal(m):
    if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear)):
        torch.nn.init.orthogonal_(m.weight, gain=2**0.5)
        torch.nn.init.zeros_(m.bias)

class ConvBackbone(torch.nn.Sequential):
    def __init__(self, c_in = 4, output_size=512):
        layers = [torch.nn.Conv2d(c_in, 32, 8, 4),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 64, 4, 2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, 3, 1),
            torch.nn.ReLU(),
            torch.nn.Flatten(),
            torch.nn.Linear(64 * 7 * 7, output_size), # filters * ((((inp_size / stride1) - 1) / stride2) - 1)
            torch.nn.ReLU()]
        for layer in layers:
            init_orthogonal(layer)
        super().__init__(*layers)

class DoubleHead(torch.nn.Module):
    def __init__(self, n_actions, inp_size=512):
        super().__init__()
        self._value_stream = torch.nn.Linear(inp_size, 1)
        init_orthogonal(self._value_stream)
        self._advantage_stream = torch.nn.Linear(inp_size, n_actions)
        init_orthogonal(self._advantage_stream)

    def forward(self, x: torch.Tensor):
        value = self._value_stream(x)
        advantage = self._advantage_stream(x)
        return (value, advantage)


class DeepNet(torch.nn.Sequential):
    def __init__(self, n_actions):
        conv_backbone = ConvBackbone()
        double_head = DoubleHead(n_actions)
        super().__init__(conv_backbone, double_head)


model = DeepNet(env.action_space.n)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [51]:
from torch.distributions import Categorical

class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].

        values, logits = self.model(inputs)
        dist = Categorical(logits=logits)
        actions = dist.sample()
        log_probs = dist.log_prob(actions)
        return {"actions": actions.data.cpu().numpy(), "logits": logits, "log_probs": log_probs, "values": values}

Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [52]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [53]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.
        latest_obs = trajectory['state']['latest_observation']
        latest_act = self.policy.act(latest_obs)
        latest_val = latest_act['values'][0]
        value_targets = [0 * len(trajectory['observations'])]
        i = len(trajectory['observations']) - 1
        for reward in reversed(trajectory['rewards']):
            value_targets[i] = reward
            if trajectory['resets'][i] or i == len(trajectory['observations']) - 1:
                latest_val = value_targets[i]
                i -= 1
                continue
                
            value_targets[i] += self.gamma * latest_val
            latest_val = value_targets[i]
            i -= 1
        
        trajectory['value_targets'] = value_targets

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [54]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        # Modify trajectory inplace.
        for key in trajectory.keys():
            value = trajectory[key]
            trajectory[key] = np.stack(value)
            


In [55]:
model = DeepNet(env.action_space.n)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [56]:
class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        log_probs = trajectory['log_probs']
        actions = trajectory['actions']
        probs = torch.softmax(trajectory['logits'], dim=-1)
        value_targets = trajectory['value_targets']
        values = trajectory['values']

        log_probs_for_actions = torch.sum(
            log_probs * F.one_hot(actions, env.action_space.n), dim=1)
        probs_for_actions = torch.sum(probs * F.one_hot(actions, env.action_space.n), dim=1)
        entropy = -1 * torch.mean(log_probs_for_actions * probs_for_actions)
        return -1 * torch.mean(log_probs_for_actions * (value_targets - values)) - self.entropy_coef * entropy

    def value_loss(self, trajectory):
        value_targets = trajectory['value_targets']
        values = trajectory['values']
        return torch.mean(torch.square(value_targets - values))

    def loss(self, trajectory):
        return self.policy_loss(trajectory) + self.value_loss(trajectory)

    def step(self, trajectory):
        self.optimizer.zero_grad()
        loss = self.loss(trajectory)
        loss.backward()
        torch.nn.utils.clip_grad_norm(self.policy.model.parameters(), max_norm=self.max_grad_norm)
        self.optimizer.step()

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.





In [65]:
%pip install tensorboard jupyter-server-proxy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [jupyter-server-proxy]
Note: you may need to restart the kernel to use updated packages.


In [64]:
#if you use TensorboardSummaries
# %load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir logs

a2c = A2C(policy, torch.optim.RMSprop(policy.model.parameters(), 7e-4, 0.99, 1e-5))


In [ ]:
<YOUR CODE: write a training loop>

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.